> **[CAPSTONE INTEGRATION TARGET]** Plato's Renderer must remain responsive while Main validates telemetry. Correlated asynchronous requests need a finite deadline, an explicit failure result, and recovery that never mistakes old sensor data for a live connection.

# Phase 3 — Invoke, await, fail, recover

Tasks: VF-300, VF-301, VF-303. Python models the trace; the TypeScript reference implements real `ipcMain.handle('plato:snapshot', ...)` and preload's `ipcRenderer.invoke`. A local fixture timeout cancels its coroutine; an Electron caller timeout does **not** cancel already running Main work.


In [ ]:
from pathlib import Path
import sys, os, json, asyncio
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts" / "notebook_support.py").exists()), None)
assert ROOT is not None, "Open this notebook inside the supplied VoltForge source bundle"
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import notebook_support as lab
LIVE = os.environ.get("VOLTFORGE_LIVE_MCP") == "1"
print(json.dumps({"event": "lab.mode", "live_opt_in": LIVE, "workspace": ROOT.name}))


## Follow the request and reply

Renderer → `window.plato.getSnapshot()` → isolated preload creates requestId → private invoke channel → Main verifies sender and payload → typed Result → preload deadline settles → Renderer updates state.

Expected errors are `INVALID_REQUEST`, `UNTRUSTED_SENDER`, `TIMEOUT`, and `INTERNAL`. Application rejection and MCP transport timeout are different layers. Record the correlated event names, not raw payloads.


In [ ]:
trace = []
store = lab.TelemetryFixture()
response = await lab.invoke_snapshot({"requestId": "phase3-normal"}, store=store, logger=trace.append)
assert response["ok"] is True
assert response["value"]["state"] == "disconnected"
assert [json.loads(row)["event"] for row in trace] == ["ipc.invoke", "ipc.resolve"]
assert {json.loads(row)["correlation"] for row in trace} == {"phase3-normal"}
print("\n".join(trace))


## Slow Main, finite caller deadline, explicit retry

The test deadline is 10 ms and simulated work takes 100 ms, keeping this exercise fast. Production preload uses a 2000 ms deadline. Retry is explicit and gets a new requestId; avoid unbounded retry loops. A late reply must not overwrite newer UI state.


In [ ]:
trace = []
slow = await lab.invoke_snapshot({"requestId": "phase3-slow"}, delay_seconds=.1,
                                 timeout_seconds=.01, logger=trace.append)
assert slow["ok"] is False and slow["error"]["code"] == "TIMEOUT"
recovered = await lab.invoke_snapshot({"requestId": "phase3-retry"}, logger=trace.append)
assert recovered["ok"] is True
assert [json.loads(row)["event"] for row in trace] == ["ipc.invoke", "ipc.timeout", "ipc.invoke", "ipc.resolve"]
print(json.dumps({"timeout": slow, "retry": recovered}, indent=2))
print("\n".join(trace))


## Receiver time determines freshness

The reference becomes stale after **1500 ms**, disconnected after **5000 ms**, and preserves clearly labeled historical values. Producer timestamps are metadata; a bad producer clock cannot make a packet fresh forever. The injected clock avoids waiting five real seconds in this lab.


In [ ]:
store = lab.TelemetryFixture()
assert store.snapshot(0)["state"] == "disconnected"
store.ingest(lab.example_packet(sequence=10), now_ms=1000)
states = [{"receiver_ms": now, "state": store.snapshot(now)["state"]}
          for now in (1000, 2500, 2501, 6000, 6001)]
assert [row["state"] for row in states] == ["live", "live", "stale", "stale", "disconnected"]
assert store.snapshot(6001)["telemetry"]["sequence"] == 10
print(json.dumps(states, indent=2))


## Bad data cannot refresh the clock

Duplicates and older sequence numbers are rejected for each retained source. The implementation remembers the latest 32 source IDs; it is a bounded ordering aid, not cryptographic replay protection. Restart a producer with a distinct session source ID or continue its monotonic sequence. An evicted source's old sequence is no longer known.


In [ ]:
for sequence in (10, 9):
    try:
        store.ingest(lab.example_packet(sequence=sequence), now_ms=6002)
        raise AssertionError("Duplicate/out-of-order packet accepted")
    except lab.ValidationError:
        pass
snapshot = store.snapshot(6002)
assert snapshot["rejected"] == 2
assert snapshot["receivedAt"] == 1000 and snapshot["state"] == "disconnected"
store.ingest(lab.example_packet(sequence=11), now_ms=6010)
assert store.snapshot(6011)["state"] == "live"
print(json.dumps(store.snapshot(6011), indent=2))


## Transport loss and application rejection are distinguishable

The fixture can model an immediate known disconnect. In the real UDP/WS reference, freshness age drives the disconnected state; the listener status separately reports starting/listening/error. Run native transport tests to observe bind failures and socket teardown.


In [ ]:
store.disconnect()
assert store.snapshot(6012)["state"] == "disconnected"
invalid = await lab.invoke_snapshot({"requestId": "x", "channel": "os:read"})
unauthorized = await lab.invoke_snapshot({"requestId": "x"}, trusted=False)
assert invalid["error"]["code"] == "INVALID_REQUEST"
assert unauthorized["error"]["code"] == "UNTRUSTED_SENDER"
print(json.dumps({"invalid": invalid, "unauthorized": unauthorized}, indent=2))


## Optional actual bridge call

Configure the discovered Renderer tool to evaluate `await window.plato.getSnapshot()` in a reviewed live session. Run with the producer stopped, then running, then paused beyond 1500/5000 ms. Use a new source session ID on restart. The notebook prints the adapter's actual result envelope; interpret it using the discovered schema, never assume a renderer-only adapter can invoke Main evaluation.

Native timeout/late reply and destroyed-window cases must be exercised by the test harness or a controlled development build. Do not expose arbitrary delay/eval knobs in the public bridge just to test them.


In [ ]:
if LIVE:
    config_path = os.environ.get("VOLTFORGE_MCP_CONFIG")
    assert config_path, "A reviewed phase-specific MCP config is required"
    native = await lab.live_mcp(lab.load_live_config(config_path))
    print(json.dumps(native, indent=2))
else:
    print(json.dumps({"event": "live.skipped", "phase": "ipc", "reason": "explicit opt-in absent"}))


## Handoff exercise

Open `../docs/phase-3/index.html`. Increase latency above the deadline; dispatch concurrent requests; disconnect, reconnect, and retry. Explain each pending promise and ensure bounded history/queue state.

Run `npm test`, `npm run build`, `npm start`, and (another terminal) `npm run telemetry:simulate` on the supported execution host. Match the native trace to this model. Submit a matrix covering success, rejection, timeout, stale, disconnect, retry, invalid telemetry, duplicate sequence, and cleanup. Mark hardware/Windows rows `not run` until observed. The next exercise is `../Projects/mini_projects/README.md`.
